# Module 15 — Exercises

Exercises for **Module 15 — Generalisation and Statistical Learning Theory**. They follow the
order of the [concept guide](README.md): Hoeffding sample sizes, Chebyshev versus Hoeffding,
the finite-class bound, growth functions and VC dimension by brute force, the optimism of
selecting the best of many models, gradient descent versus the pseudo-inverse, a double-descent
experiment, two conceptual questions, and a mini-project that fits a power law to the
generalisation gap. Use only NumPy and the standard library (plus `scipy` where a prompt says
so) and keep everything seeded.

**How these exercises work.** Each coding exercise has a *starter* cell (replace `# YOUR CODE HERE` with your implementation) followed by a *check* cell. Run the check cell to get an instant ✅/❌. Checks never stop the notebook, so you can run everything top-to-bottom at any time and see your score at the end. Some checks compare against a *hidden* answer (only a hash is stored), so you cannot peek at the number — think first, then verify. Conceptual questions hide their answer behind a *Show answer* toggle. Mini-projects are open-ended and come with a rubric. Difficulty is marked ★☆☆ (warm-up), ★★☆ (core), ★★★ (stretch).

### Setup and self-check helpers
Run the next two cells first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, log, sqrt, exp
rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (7, 4)
np.set_printoptions(precision=4, suppress=True)

In [ ]:
import hashlib as _hashlib
import numpy as _np

_results = {}

def _canon(value, decimals=3):
    """Canonical string form of an answer, so hashes are stable."""
    if isinstance(value, (bool, _np.bool_)):
        return "true" if value else "false"
    if isinstance(value, str):
        return " ".join(value.strip().lower().split())
    if isinstance(value, (int, _np.integer)):
        return str(int(value))
    if isinstance(value, (float, _np.floating)):
        s = f"{float(value):.{decimals}f}"
        return "0." + "0" * decimals if s in ("-0." + "0" * decimals, "-0") else s
    if isinstance(value, dict):
        return "{" + ",".join(f"{_canon(k, decimals)}:{_canon(v, decimals)}" for k, v in sorted(value.items(), key=lambda kv: str(kv[0]))) + "}"
    try:
        arr = _np.asarray(value)
        if arr.ndim == 0:
            return _canon(arr.item(), decimals)
        return "[" + ",".join(_canon(x, decimals) for x in arr.ravel().tolist()) + "]"
    except Exception:
        return str(value)

def sha_of(value, decimals=3):
    return _hashlib.sha256(_canon(value, decimals).encode()).hexdigest()[:16]

def check(name, condition, hint=""):
    """Record and print a pass/fail mark. Never raises, so the notebook keeps running."""
    try:
        ok = bool(condition)
    except Exception:
        ok = False
    _results[name] = ok
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"   — hint: {hint}"))
    return ok

def check_close(name, got, expected, tol=1e-6, hint=""):
    try:
        g = _np.asarray(got, dtype=float)
        e = _np.asarray(expected, dtype=float)
        ok = g.shape == e.shape and bool(_np.allclose(g, e, atol=tol, rtol=tol))
    except Exception:
        ok = False
    return check(name, ok, hint)

def check_hash(name, value, expected_sha, decimals=3, hint=""):
    """Compare an answer against a hidden expected value (only its hash is stored)."""
    try:
        ok = value is not None and sha_of(value, decimals) == expected_sha
    except Exception:
        ok = False
    return check(name, ok, hint)

def summary():
    n_ok = sum(1 for v in _results.values() if v)
    n = len(_results)
    print(f"\nScore: {n_ok}/{n} checks passed" + ("  🎉" if n and n_ok == n else ""))
    if n and n_ok < n:
        print("Not yet passing: " + ", ".join(k for k, v in _results.items() if not v))

## Exercise 1 — Hoeffding sample size and deviation  `★☆☆`

For a *fixed* classifier evaluated on $n$ i.i.d. test examples, Hoeffding's inequality gives,
with probability at least $1 - \delta$, $|\hat a - a| \le \sqrt{\ln(2/\delta)/(2n)}$.

Implement

* `hoeffding_n(eps, delta)` — the smallest **integer** $n$ (use `np.ceil`, return a Python `int`)
  guaranteeing $|\hat a - a| \le \varepsilon$ with probability $\ge 1-\delta$;
* `hoeffding_eps(n, delta)` — the deviation $\varepsilon$ guaranteed by $n$ examples (a `float`).

Then set `n_benchmark = hoeffding_n(0.01, 0.01)` (a benchmark that pins accuracy to one point
at 99 %) and `eps_benchmark = hoeffding_eps(5000, 0.05)`.

In [ ]:
def hoeffding_n(eps, delta):
    # YOUR CODE HERE
    return None

def hoeffding_eps(n, delta):
    # YOUR CODE HERE
    return None

n_benchmark = None     # YOUR CODE HERE
eps_benchmark = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    got_n = hoeffding_n(0.02, 0.05)
except Exception:
    got_n = None
check('n for eps = 0.02, delta = 0.05 is 4612', got_n == 4612, hint="ln(40) / (2 * 0.0004), rounded up")
try:
    got_eps = hoeffding_eps(4612, 0.05)
except Exception:
    got_eps = None
check('eps for n = 4612, delta = 0.05 is just under 0.02', got_eps is not None and 0.0199 < got_eps <= 0.02)
check_hash('hidden: n_benchmark', n_benchmark if isinstance(n_benchmark, (int, np.integer)) else None, '68206c4949fc2c4e')
check_hash('hidden: eps_benchmark (3 decimals)', eps_benchmark if isinstance(eps_benchmark, float) else None, 'b654f6a8a45bb365')

## Exercise 2 — Chebyshev versus Hoeffding  `★☆☆`

For a sample proportion $\hat p$ of $n$ Bernoulli$(p)$ trials, two bounds on $P(|\hat p - p| \ge t)$:

$$\text{Chebyshev: } \frac{p(1-p)}{n t^2}, \qquad \text{Hoeffding: } 2e^{-2nt^2}.$$

Implement `chebyshev_tail(n, t, p)` and `hoeffding_tail(n, t)` (both capped at 1 with `min`,
returning `float`), and `crossover_n(t, p)` — the smallest integer $n \ge 1$ at which the
Hoeffding bound becomes **strictly smaller** than the Chebyshev bound (loop upward from 1).

Then set `ratio_1000 = chebyshev_tail(1000, 0.05, 0.8) / hoeffding_tail(1000, 0.05)` and
`n_cross = crossover_n(0.05, 0.8)`.

In [ ]:
def chebyshev_tail(n, t, p):
    # YOUR CODE HERE
    return None

def hoeffding_tail(n, t):
    # YOUR CODE HERE
    return None

def crossover_n(t, p):
    # YOUR CODE HERE
    return None

ratio_1000 = None   # YOUR CODE HERE
n_cross = None      # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    c = chebyshev_tail(100, 0.1, 0.8); h = hoeffding_tail(100, 0.1)
except Exception:
    c = h = None
check_close('Chebyshev n=100, t=0.1, p=0.8 is 0.16', c, 0.16)
check_close('Hoeffding n=100, t=0.1 is 2 exp(-2)', h, 2 * np.exp(-2))
try:
    capped = chebyshev_tail(10, 0.01, 0.5)
except Exception:
    capped = None
check('bounds are capped at 1', capped == 1.0)
check_hash('hidden: ratio Chebyshev/Hoeffding at n=1000, t=0.05 (3 decimals)', ratio_1000 if isinstance(ratio_1000, float) else None, 'ec9abb00be7375c0')
check_hash('hidden: crossover n for t=0.05, p=0.8', n_cross if isinstance(n_cross, (int, np.integer)) else None, 'de0023e398111d43')

## Exercise 3 — The finite-class uniform bound  `★☆☆`

Union bound + Hoeffding: with probability $\ge 1 - \delta$, **every** $h$ in a finite class
$\mathcal H$ satisfies $|R(h) - \hat R_S(h)| \le \sqrt{\ln(2|\mathcal H|/\delta)/(2n)}$.

Implement `finite_class_bound(n, size_H, delta)` (a `float`) and `finite_class_n(eps, size_H, delta)`
— the smallest integer $n$ that makes the bound $\le \varepsilon$ (a Python `int`).

Then: decision stumps on 20 features with 100 thresholds and 2 orientations each form a class
of `n_stumps = 20 * 100 * 2` hypotheses. Set `bound_stumps = finite_class_bound(2000, n_stumps, 0.05)`
and `n_needed_million = finite_class_n(0.03, 10**6, 0.01)`.

In [ ]:
def finite_class_bound(n, size_H, delta):
    # YOUR CODE HERE
    return None

def finite_class_n(eps, size_H, delta):
    # YOUR CODE HERE
    return None

n_stumps = 20 * 100 * 2
bound_stumps = None        # YOUR CODE HERE
n_needed_million = None    # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    b1 = finite_class_bound(1000, 1, 0.05)
except Exception:
    b1 = None
check_close('|H| = 1 recovers Hoeffding: sqrt(ln(40)/2000)', b1, np.sqrt(np.log(40) / 2000))
try:
    n1 = finite_class_n(0.02, 1, 0.05)
except Exception:
    n1 = None
check('|H| = 1, eps = 0.02 needs 4612 examples', n1 == 4612)
try:
    bigger = finite_class_bound(1000, 10**6, 0.05) > finite_class_bound(1000, 10**3, 0.05)
except Exception:
    bigger = False
check('a larger class gives a larger bound', bigger)
check_hash('hidden: bound for 4,000 stumps, n = 2000 (3 decimals)', bound_stumps if isinstance(bound_stumps, float) else None, 'c945fbec264791e3')
check_hash('hidden: n needed for a million hypotheses, eps = 0.03, delta = 0.01', n_needed_million if isinstance(n_needed_million, (int, np.integer)) else None, 'f8a7ecdfa85fd8ba')

## Exercise 4 — Growth function by brute force: thresholds  `★★☆`

The growth function $\Pi_{\mathcal H}(n)$ counts the distinct labellings (dichotomies) a class
realises on $n$ points. Implement

* `growth_function(hypotheses, points)` — `hypotheses` is a list of callables `h(x) -> bool`;
  return the number of **distinct** tuples `(h(x_1), ..., h(x_n))` over the list;
* `threshold_hypotheses(points)` — a list of threshold rules $h_a(x) = \mathbf 1[x \ge a]$ that
  together realise every dichotomy thresholds can produce on `points` (hint: cut points below
  the smallest point, between consecutive sorted points, and above the largest — one rule each;
  bind `a` as a default argument, `lambda x, a=a: x >= a`);
* `threshold_growth(n)` — the growth function of thresholds on `n` distinct points.

Set `growth_thresholds_12 = threshold_growth(12)`. Every function should work for any
distinct real-valued points.

In [ ]:
from itertools import product

def growth_function(hypotheses, points):
    # YOUR CODE HERE  (collect label tuples in a set, return its size)
    return None

def threshold_hypotheses(points):
    # YOUR CODE HERE  (one lambda per cut point; use `lambda x, a=a: x >= a`)
    return None

def threshold_growth(n):
    # YOUR CODE HERE
    return None

growth_thresholds_12 = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    g3 = threshold_growth(3); g5 = threshold_growth(5)
except Exception:
    g3 = g5 = None
check('thresholds realise n + 1 dichotomies on 3 points', g3 == 4)
check('thresholds realise n + 1 dichotomies on 5 points', g5 == 6)
try:
    pts = np.array([0.3, -1.2, 4.0, 2.2])
    const = growth_function([lambda x: True, lambda x: False, lambda x: x > -100], pts)
except Exception:
    const = None
check('growth_function counts DISTINCT dichotomies (two constant rules + a duplicate -> 2)', const == 2)
check_hash('hidden: growth function of thresholds on 12 points', growth_thresholds_12 if isinstance(growth_thresholds_12, (int, np.integer)) else None, '3fdba35f04dc8c46')

## Exercise 5 — Shattering and the VC dimension of intervals  `★★☆`

The class of intervals $h_{a,b}(x) = \mathbf 1[a \le x \le b]$ on sorted points realises a
labelling exactly when its 1s form a single contiguous block (or there are none).

Implement

* `interval_realisable(labels)` — `labels` is a sequence of 0/1 for points **already sorted by
  position**; return `True` iff some interval realises it;
* `interval_growth(n)` — the number of realisable labellings of $n$ points (iterate over
  `product([0, 1], repeat=n)` from `itertools`);
* `interval_vc_dimension(max_n=8)` — the largest $n \le$ `max_n` with `interval_growth(n) == 2**n`.

Set `vc_intervals = interval_vc_dimension()` and `growth_intervals_10 = interval_growth(10)`.
The closed form you should recover is $\Pi(n) = \binom{n}{2} + n + 1$.

In [ ]:
from itertools import product

def interval_realisable(labels):
    # YOUR CODE HERE  (the 1s must form one contiguous block, or be absent)
    return None

def interval_growth(n):
    # YOUR CODE HERE
    return None

def interval_vc_dimension(max_n=8):
    # YOUR CODE HERE
    return None

vc_intervals = None          # YOUR CODE HERE
growth_intervals_10 = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    r1 = interval_realisable([0, 1, 1, 0]); r2 = interval_realisable([1, 0, 1]); r3 = interval_realisable([0, 0, 0])
except Exception:
    r1 = r2 = r3 = None
check('0110 is realisable by an interval', r1 is True)
check('101 is NOT realisable (needs two intervals)', r2 is False)
check('all-zero labelling is realisable (empty interval)', r3 is True)
try:
    g3 = interval_growth(3); g6 = interval_growth(6)
except Exception:
    g3 = g6 = None
check('growth function on 3 points is 7', g3 == 7)
check('growth function matches C(n,2) + n + 1 at n = 6', g6 == comb(6, 2) + 6 + 1)
check_hash('hidden: VC dimension of intervals', vc_intervals if isinstance(vc_intervals, (int, np.integer)) else None, 'd4735e3a265e16ee')
check_hash('hidden: growth function on 10 points', growth_intervals_10 if isinstance(growth_intervals_10, (int, np.integer)) else None, '7688b6ef52555962')

## Exercise 6 — Simulating the winner's curse  `★★☆`

Implement `best_of_m_optimism(m, n, acc, n_rep, g)`: simulate `n_rep` rounds in which $m$ models
with the **same** true accuracy `acc` are each scored on an independent validation set of `n`
examples (`g.binomial(n, acc, size=(n_rep, m)) / n` draws all scores at once; `g` is a
`numpy.random.Generator`), keep the best score in each round, and return the average of
`best - acc` as a `float`.

Then build the dictionary `opt = {m: best_of_m_optimism(m, 1000, 0.8, 4000, np.random.default_rng(1)) for m in (1, 10, 100)}`.
The theory predicts about $0$, $1.54\cdot\text{SE}$ and $2.51\cdot\text{SE}$ with
$\text{SE} = \sqrt{0.8 \cdot 0.2/1000} \approx 0.0126$.

In [ ]:
def best_of_m_optimism(m, n, acc, n_rep, g):
    # YOUR CODE HERE
    return None

opt = None   # YOUR CODE HERE  ({m: ... for m in (1, 10, 100)})

**Check your answer:**

In [ ]:
try:
    o1 = opt[1]; o10 = opt[10]; o100 = opt[100]
except Exception:
    o1 = o10 = o100 = None
se = np.sqrt(0.8 * 0.2 / 1000)
check('m = 1: no optimism (|mean - acc| < 0.002)', o1 is not None and abs(o1) < 0.002)
check('m = 10: optimism within 20% of 1.54 SE', o10 is not None and 0.8 * 1.539 * se < o10 < 1.2 * 1.539 * se)
check('m = 100: optimism within 20% of 2.51 SE', o100 is not None and 0.8 * 2.508 * se < o100 < 1.2 * 2.508 * se)
check('optimism grows with m', None not in (o1, o10, o100) and o1 < o10 < o100)
try:
    big_n = best_of_m_optimism(10, 100_000, 0.8, 500, np.random.default_rng(2))
except Exception:
    big_n = None
check('a 100x larger validation set shrinks the optimism ~10x', big_n is not None and big_n < 0.004)

## Exercise 7 — Gradient descent from zero equals the pseudo-inverse  `★★☆`

Implement `gd_least_squares(X, y, steps=5000, lr=None)`: run plain gradient descent on
$\tfrac12\|Xw - y\|^2$ **starting from `w = 0`**, with step size `lr` (default
`1 / np.linalg.norm(X, 2)**2`, i.e. $1/\sigma_{\max}^2$), for `steps` iterations, and return `w`.
Use only NumPy array operations — no `lstsq`, `pinv` or `solve`.

The checks verify that for $p > n$ your iterate (i) interpolates the data, (ii) coincides with
`np.linalg.pinv(X) @ y`, and (iii) is shorter than any other interpolator; and that for $p < n$
it recovers ordinary least squares.

In [ ]:
def gd_least_squares(X, y, steps=5000, lr=None):
    # YOUR CODE HERE  (w = 0; repeat w <- w - lr * X^T (X w - y))
    return None

**Check your answer:**

In [ ]:
_g = np.random.default_rng(7)
_X = _g.normal(size=(20, 50)); _y = _X @ _g.normal(size=50)          # p > n, noiseless
try:
    _w = gd_least_squares(_X, _y)
except Exception:
    _w = None
check('p > n: GD interpolates the data', _w is not None and np.allclose(_X @ _w, _y, atol=1e-6))
check_close('p > n: GD from zero equals pinv(X) y', _w, np.linalg.pinv(_X) @ _y, tol=1e-6)
try:
    _other = _w + (np.eye(50) - np.linalg.pinv(_X) @ _X) @ _g.normal(size=50)   # add a null-space vector
    shorter = np.linalg.norm(_w) < np.linalg.norm(_other) and np.allclose(_X @ _other, _y, atol=1e-6)
except Exception:
    shorter = False
check('p > n: the GD solution is the shortest interpolator', shorter)
_X2 = _g.normal(size=(60, 5)); _y2 = _X2 @ np.arange(1., 6.) + 0.1 * _g.normal(size=60)
try:
    _w2 = gd_least_squares(_X2, _y2)
except Exception:
    _w2 = None
check_close('p < n: GD recovers ordinary least squares', _w2, np.linalg.lstsq(_X2, _y2, rcond=None)[0], tol=1e-5)

## Exercise 8 — A double-descent experiment  `★★★`

Reproduce double descent with random ReLU features. Implement

* `relu_features(X, W, b)` returning $\max(0, XW + b)$ (shape `(n, p)`);
* `double_descent_errors(n=40, ps=(20, 40, 200), reps=10, sigma=0.3, seed=0)`: with
  `g = np.random.default_rng(seed)`, repeat `reps` times — draw `X_tr` of shape `(n, 5)` from
  `g.normal`, labels `y_tr = target(X_tr) + sigma * g.normal(size=n)` with
  `target(X) = np.sin(X[:, 0]) + 0.5 * X[:, 1] * X[:, 2]`, a noiseless test set `X_te` of 400 rows
  (`y_te = target(X_te)`), then `W = g.normal(size=(5, max(ps)))` and `b = g.uniform(-1, 1, max(ps))`
  (draw in exactly this order); for each `p` in `ps` use the **first** `p` columns of `W` / entries
  of `b`, fit the output weights by `np.linalg.lstsq(Phi, y_tr, rcond=None)[0]` (minimum-norm),
  and record the test MSE. Return `{p: median test MSE over reps}`.

Set `dd = double_descent_errors()`. The checks test the *shape*: the error at $p = n$ (the
interpolation threshold) must be far above both $p = n/2$ and $p = 5n$, and $p = 5n$ must beat
$p = n/2$ — the second descent.

In [ ]:
def relu_features(X, W, b):
    # YOUR CODE HERE
    return None

def double_descent_errors(n=40, ps=(20, 40, 200), reps=10, sigma=0.3, seed=0):
    # YOUR CODE HERE
    return None

dd = None   # YOUR CODE HERE

**Check your answer:**

In [ ]:
try:
    e_half, e_n, e_5n = dd[20], dd[40], dd[200]
except Exception:
    e_half = e_n = e_5n = None
check('under-parameterised (p = n/2): test MSE below 0.8', e_half is not None and 0.1 < e_half < 0.8)
check('interpolation threshold (p = n): test MSE at least 5x the p = n/2 value', None not in (e_half, e_n) and e_n > 5 * e_half)
check('over-parameterised (p = 5n): test MSE below 0.6', e_5n is not None and e_5n < 0.6)
check('second descent: p = 5n beats p = n/2', None not in (e_half, e_5n) and e_5n < e_half)
try:
    Phi = relu_features(np.array([[1., -2.]]), np.array([[1., 0.], [0., 1.]]), np.array([0., 0.]))
except Exception:
    Phi = None
check_close('relu_features clips negatives: [[1, -2]] -> [[1, 0]]', Phi, np.array([[1., 0.]]))

## Exercise 9 — Why VC theory does not explain deep learning  `★★☆`

A ReLU network with $W = 2.5\times10^7$ weights and $L = 50$ layers has VC dimension of order
$WL\log W \approx 3\times10^{10}$. It is trained on $1.28\times10^6$ images, reaches zero training
error, and scores 76 % top-1 accuracy on the test set. A colleague says: "VC theory predicts
this network cannot generalise, so the theory is wrong."

Explain (a) what the VC bound actually says here and why that is *not* a prediction of failure;
(b) which experimental fact (Zhang et al., 2017) shows that *any* uniform-convergence argument
over the whole architecture class must be vacuous; (c) two ingredients the modern account adds
to recover a non-vacuous story.

<details><summary><b>Show answer</b></summary>

(a) The VC bound reads $R(h) \le \hat R_S(h) + \text{(something} \gtrsim \sqrt{d_{\mathrm{VC}}/n})$, and with
$d_{\mathrm{VC}}/n \approx 2\times10^4$ the right-hand side exceeds 1. A bound above 1 is *vacuous*, not
false: it says the theory offers no guarantee, which is different from predicting failure. The
bound is a worst case over all distributions and all hypotheses in the class; it cannot be
tightened for this architecture because (b) the same network reaches zero training error on
CIFAR-10 with randomly permuted labels. If the class can fit arbitrary labels on $n$ points,
there *are* hypotheses in it whose generalisation gap is essentially 1, so any statement of the
form "for all $h$ in the class, $R(h) - \hat R_S(h)$ is small" is simply untrue; uniform convergence
over the architecture class is dead on arrival. (c) The modern account restricts attention to
the hypotheses the *algorithm actually produces*: implicit regularisation (gradient descent from
small initialisation prefers low-norm, low-complexity, large-margin solutions — the min-norm
result for least squares, max-margin for logistic loss), so the effective class is far smaller
than the architecture class and norm-based or margin-based bounds replace parameter counts; and
benign overfitting / double descent (in the over-parameterised regime the interpolator can
absorb label noise in many weak directions while fitting the signal in a few strong ones). A
third ingredient is algorithmic stability or PAC-Bayes bounds, which are about the learned
distribution over weights rather than the worst member of the class. The honest summary is
that VC theory is correct but answers a question ("what holds for every function the
architecture can represent?") that is not the one deep learning poses.

</details>

## Exercise 10 — What no free lunch does and does not say  `★★☆`

Two claims, both citing the no-free-lunch theorem:

1. "Averaged over all possible target functions, no algorithm beats random guessing on unseen
   inputs — so benchmarking learning algorithms against each other is meaningless."
2. "Since convolutional networks beat fully connected networks on images, the no-free-lunch
   theorem must be false."

Diagnose each claim. Then explain, with one concrete example, what an inductive bias is and
why its *mismatch* with a problem is the mechanism by which NFL bites in practice. Finally,
state which assumption of this entire module is itself an inductive bias and name one way it
fails in deployment.

<details><summary><b>Show answer</b></summary>

Claim 1 is a misreading of the averaging. NFL averages uniformly over *all* $2^{|\mathcal X|}$
labelings of the domain, almost all of which are incompressible noise with no relation between
nearby inputs. Real problems are not drawn from that uniform measure — images, text and
molecules have structure — and on any *particular* distribution algorithms differ enormously.
Benchmarks measure performance on the structured problems we care about, which is exactly
what NFL leaves unconstrained. Claim 2 is the mirror error: NFL is a theorem about the uniform
average, and a CNN's advantage on images is consistent with it because a CNN must be
correspondingly *worse* than a fully connected network on the (never-seen, unnatural) targets
whose labels change arbitrarily under translation. The theorem says every advantage on some
targets is paid for on others; it does not say the advantages do not exist. An inductive bias
is the set of assumptions that let a learner prefer some hypotheses over others consistent
with the data: a convolution assumes the same local pattern means the same thing anywhere in
the image (translation equivariance, locality), which is why it needs 36,864 weights for a
$3\times3$, 64-to-64-channel layer instead of billions — and why it will fail on a task where
the position of a pattern is what matters, unless positional information is added back. NFL
bites exactly when the bias is wrong for the problem: a smoothness prior on a discontinuous
target, a bag-of-words model on a task where word order carries the meaning, a linear model
on XOR. The module-wide assumption that is itself a bias is that training and deployment data
are i.i.d. draws from the same distribution $P$. It fails under covariate shift (a different
case mix at a new hospital), label shift (a different prevalence), or concept shift (the
meaning of the features changes, as in fraud patterns after a policy change) — and every bound
in the module is silent the moment it does.

</details>

## Exercise 11 — Mini-project: the generalisation gap as a power law  `★★★`

Scaling laws say loss falls as a power of data; classical theory says the generalisation gap
of a well-specified $p$-parameter model falls like $p\sigma^2/n$. Measure one and compare with
the other.

1. For each $n$ in `ns`, repeat `n_reps` times: draw $x \sim \text{Uniform}[-1, 1]$, labels
   $y = f(x) + \sigma\varepsilon$; fit a degree-3 polynomial by least squares (`np.polyfit`); record
   the training MSE, the test MSE on `n_test` fresh noisy points, and their difference (the gap).
2. Average over repetitions and plot test MSE and training MSE against $n$ (log $x$-axis) with
   the noise floor $\sigma^2$ marked; on a second log–log panel plot the mean gap against $n$.
3. Fit a power law $\text{gap} \approx c\,n^{-\gamma}$ by linear regression of $\log(\text{gap})$ on
   $\log n$; report $c$ and $\gamma$.
4. Compare with theory: for OLS with $p$ parameters, $\mathbb E[\text{train MSE}] \approx \sigma^2(1 - p/n)$
   and $\mathbb E[\text{test MSE}] \approx \sigma^2(1 + p/n)$, so the gap is $\approx 2\sigma^2 p/n$:
   $\gamma = 1$ and $c = 2\sigma^2 p = 2.0$. Overlay this line.
5. Fit the Chinchilla-style form $\text{test MSE} \approx E + B\,n^{-\beta}$ with
   `scipy.optimize.curve_fit` and compare $E$ with $\sigma^2$ and $\beta$ with $\gamma$.

The cell below fixes the setting. Keep the total runtime modest (the defaults take a few seconds).

**Rubric — you have mastered this when you can:**

- Training MSE rises towards $\sigma^2$ and test MSE falls towards it from above; both are
  plotted against $n$ with the noise floor marked, with titles, axis labels and a legend.
- The fitted exponent $\gamma$ is close to $1$ (roughly $0.9$–$1.2$) and you can say *why* it is
  $1$ and not $1/2$ (variance of a well-specified estimator scales like $1/n$; the $1/\sqrt n$ of
  Hoeffding is a worst-case *bound* on deviations, not the expected gap).
- The fitted constant $c$ is of the order of $2\sigma^2 p = 2$ (the realised value is somewhat
  larger because the uniform design makes $\mathbb E[x^\top(X^\top X)^{-1}x]$ exceed $p/n$ at small $n$).
- The scaling-law fit returns $E \approx \sigma^2 = 0.25$ — the irreducible error, i.e. the
  Bayes risk — and you can map $E$, $B n^{-\beta}$ onto the Chinchilla terms $E$, $B/D^\beta$.
- You can state what changes if the degree is *too low* (test MSE plateaus above $\sigma^2$:
  approximation error; the gap still shrinks) and *too high* (gap larger, same exponent).
- (Bonus) Repeat with degree 1 and degree 8 and overlay the three gap curves.

In [ ]:
# ---- setting: cubic target, polynomial regression of the correct degree, noise sigma = 0.5 ----
g = np.random.default_rng(0)
coef_true = np.array([2.0, 0.0, -1.0, 0.5])        # highest power first (np.polyval convention): 2x^3 - x + 0.5
f_true = lambda x: np.polyval(coef_true, x)
sigma, degree = 0.5, 3
ns = np.array([20, 40, 80, 160, 320, 640, 1280])
n_reps, n_test = 200, 2000
print("number of parameters p =", degree + 1, "; noise variance sigma^2 =", sigma ** 2)


# YOUR CODE HERE
# for n in ns: repeat n_reps times -> train MSE, test MSE, gap; average
# plot test/train MSE vs n (semilogx) with sigma^2 marked; plot gap vs n (loglog)
# fit log(gap) = log(c) - gamma * log(n); overlay 2 sigma^2 p / n
# fit test MSE ~ E + B n^(-beta) with scipy.optimize.curve_fit

## Score

In [ ]:
summary()